<a href="https://colab.research.google.com/github/pongfei/Contact0517/blob/main/confluent_kafka_producer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# send a message to confluent-kafka topic via python
# customer_id as key & evetything else details

In [ ]:
# Required connection configs for Kafka producer, consumer, and admin
bootstrap.servers=pkc-619z3.us-east1.gcp.confluent.cloud:9092
security.protocol=SASL_SSL
sasl.mechanisms=PLAIN
sasl.username=D45OSUTAUNNNESFE
sasl.password=cfltLJrIohOKYsqIcq9Fu5cTB+/hp2jXWb3msVsj8tKa27e7puNinySlu7cayzgA

# Best practice for higher availability in librdkafka clients prior to 1.7
session.timeout.ms=45000

client.id=ccloud-python-client-09893e13-6dfb-4e40-b6f7-05bfc6d0ff21


In [ ]:
!pip install confluent-kafka

In [ ]:
import pandas as pd
import json

csv_file = 'Customers_data/first_100_customers.csv'

df = pd.read_csv(csv_file)

df.head()


,customer_id,name,city,state,country,registration_date,is_active
0,0,Customer_0,Pune,Maharashtra,India,2023-06-29,False
1,1,Customer_1,Bangalore,Tamil Nadu,India,2023-12-07,True
2,2,Customer_2,Hyderabad,Gujarat,India,2023-10-27,True
3,3,Customer_3,Bangalore,Karnataka,India,2023-10-17,False
4,4,Customer_4,Ahmedabad,Karnataka,India,2023-03-14,False


In [ ]:
#change to JSON
json_records = df.to_dict(orient = 'records')

json_file = 'customers.json'

with open(json_file,'w') as file:
  json.dump(json_records,file,indent=4)


print("File converted to JSON")

File converted to JSON


In [ ]:
from confluent_kafka import Producer
import json
import time

conf = {
"bootstrap.servers":"pkc-619z3.us-east1.gcp.confluent.cloud:9092",
"security.protocol":"SASL_SSL",
"sasl.mechanisms":"PLAIN",
"sasl.username":"D45OSUTAUNNNESFE",
"sasl.password":"cfltLJrIohOKYsqIcq9Fu5cTB+/hp2jXWb3msVsj8tKa27e7puNinySlu7cayzgA",
    "session.timeout.ms":45000,
    "client.id":"ccloud-python-client-09893e13-6dfb-4e40-b6f7-05bfc6d0ff21"
    }
producer = Producer(conf)

In [ ]:
topic = 'ecommerce'

with open('customers.json','r') as file:
  customers_data = json.load(file)

value = customers_data[0]
key = value['customer_id']

print(key,value)

0 {'customer_id': 0, 'name': 'Customer_0', 'city': 'Pune', 'state': 'Maharashtra', 'country': 'India', 'registration_date': '2023-06-29', 'is_active': False}


In [ ]:
#convert to byte first (.produce only takes bytes)
type(str(value).encode('utf-8'))

bytes

In [ ]:
# work on sending a single value first then loop

producer.produce(topic,key = str(key).encode('utf-8'),value=str(value).encode('utf-8'))

In [ ]:
#send multiple messages to kafka cluster

def delivery_status(err, msg):
    if err:
        print(f"Message delivery failed: {err}")
    else:
        print(f"Message delivered to {msg.topic()} [{msg.partition()}] at offset {msg.offset()}")

for record in customers_data:
    try:
        message_value = json.dumps(record)
        message_key = str(record['customer_id']).encode('utf-8')

        producer.produce(topic, key=message_key, value=message_value, callback=delivery_status)
        producer.poll(1)

    except Exception as e:
        print(f"Error sending message: {e}")

producer.flush()

print("message sent to Kafka successfully")


Message delivered to ecommerce [2] at offset 5
Message delivered to ecommerce [2] at offset 6
Message delivered to ecommerce [1] at offset 0
Message delivered to ecommerce [1] at offset 1
Message delivered to ecommerce [1] at offset 2
Message delivered to ecommerce [1] at offset 3
Message delivered to ecommerce [1] at offset 4
Message delivered to ecommerce [0] at offset 1
Message delivered to ecommerce [2] at offset 7
Message delivered to ecommerce [0] at offset 2
Message delivered to ecommerce [0] at offset 3
Message delivered to ecommerce [0] at offset 4
Message delivered to ecommerce [0] at offset 5
Message delivered to ecommerce [2] at offset 8
Message delivered to ecommerce [0] at offset 6
Message delivered to ecommerce [1] at offset 5
Message delivered to ecommerce [0] at offset 7
Message delivered to ecommerce [2] at offset 9
Message delivered to ecommerce [0] at offset 8
Message delivered to ecommerce [1] at offset 6
Message delivered to ecommerce [0] at offset 9
Message deliv